# 04. FastMCP & Agentic Skills 아키텍처 구축 실무

> **핵심 학습 목표**: 사내 이종 데이터 소스를 **FastMCP 마이크로서비스**로 격리 구축하고, 엔터프라이즈 보안(StaticToken, Custom Claims, RSA/JWT)과 서버 합성을 적용한 뒤, **LangChain `create_agent`** 및 **Anthropic 표준 Skills 패턴(점진적 공개)**을 결합하여 실무급 에이전틱 아키텍처를 완성합니다.

---

### 💡 핵심 학습 로드맵
```
[1. FastMCP 기초 & 엔터프라이즈 보안]
  • Tool / Resource Primitive & Pydantic 스키마 검증
  • StaticTokenVerifier (Scope 기반 권한 분리)
  • 금융권 Custom Claims 인가 (부서/직급 팩토리 함수)
  • 가상 IdP RSA 비대칭키 JWT 서명 및 검증
  • Server Composition (부서별 서버 -> Enterprise Gateway 마운트)
           │
           ▼
[2. 연동 패러다임 비교: 정적 주입 vs 동적 Skills]
  • 방식 A (langchain_mcp_adapters): MultiServerMCPClient 정적 일괄 바인딩 (Tool Explosion 한계 체험)
  • 방식 B (Anthropic Skills 패턴): 범용 원시 도구(tools/common.py) + skills/mcp 동적 점진적 공개(Progressive Disclosure)
```


---
# 0. 환경 설정 및 서버 프로세스 관리

로컬 루트의 `.env` 파일로부터 API Key를 로드하고, 포트 충돌을 방지하기 위한 동적 포트 탐색 및 백그라운드 서버 프로세스 관리 유틸리티를 설정합니다.


In [ ]:
import os
import sys
from dotenv import load_dotenv
from IPython.display import display, Markdown

# 프로젝트 루트를 sys.path에 추가
PROJECT_ROOT = os.path.abspath('..') if os.path.exists('../rag') else os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

ENV_PATH = os.path.join(PROJECT_ROOT, '.env')
if os.path.exists(ENV_PATH):
    load_dotenv(ENV_PATH, override=True)
    print(f'✅ 환경변수 로드 완료: {ENV_PATH}')
else:
    load_dotenv(override=True)


### 가용 포트 탐색 및 백그라운드 프로세스 유틸리티

로컬 실습 환경에서 포트 충돌을 방지하고 여러 개의 FastMCP 서버를 안전하게 기동/종료할 수 있도록 관리 헬퍼를 정의합니다.


In [ ]:
import socket
import subprocess
import time
import signal

# 활성화된 백그라운드 서버 프로세스 관리 딕셔너리
ACTIVE_SERVERS = {}

def find_available_port(start=6000, end=7500):
    """지정된 범위 내에서 바인딩 가능한 빈 포트를 탐색합니다."""
    for port in range(start, end):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('0.0.0.0', port))
                return port
            except OSError:
                continue
    raise RuntimeError("사용 가능한 포트를 찾지 못했습니다.")

def start_mcp_server(
        name: str,
        script_path: str,
        port: int,
        extra_args: list = None,
        restart: bool = True,
    ) -> str:
      """FastMCP 서버를 기동합니다.
      이미 실행 중인 동일 이름의 서버가 있다면 안전하게 종료 후 새 설정으로 재기동합니다.
      """
      # 1. 이미 동일 이름의 서버가 실행 중이면 자동 종료 (Hot Reload)
      if name in ACTIVE_SERVERS:
        old_info = ACTIVE_SERVERS[name]
        proc = old_info["process"]
        if proc.poll() is None:
          proc.terminate()
          try:
            proc.wait(timeout=2)
          except subprocess.TimeoutExpired:
            proc.kill()
          print(
              f"🔄 [{name}] 이전 프로세스(PID: {proc.pid}, Port:"
              f" {old_info['port']})를 종료하고 재기동합니다."
          )
        del ACTIVE_SERVERS[name]
      # 2. 새 프로세스 기동
      cmd = [sys.executable, "-u", script_path, "--port", str(port)]
      if extra_args:
        cmd.extend(extra_args)
      log_file = open(f"{script_path}.log", "w", encoding="utf-8")
      proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
      time.sleep(2)  # 서버 기동 대기
      url = f"http://localhost:{port}/mcp"
      ACTIVE_SERVERS[name] = {
          "process": proc,
          "port": port,
          "url": url,
          "log": f"{script_path}.log",
      }
      print(f"🚀 [{name}] FastMCP 서버 기동 완료 -> {url} (PID: {proc.pid})")
      return url

def stop_all_servers():
    """기동된 모든 백그라운드 FastMCP 서버 프로세스를 정상 종료합니다."""
    for name, info in list(ACTIVE_SERVERS.items()):
        proc = info["process"]
        if proc.poll() is None:
            proc.terminate()
            proc.wait(timeout=3)
            print(f"🛑 [{name}] 서버(PID: {proc.pid}, Port: {info['port']})가 안전하게 종료되었습니다.")
    ACTIVE_SERVERS.clear()

print("✅ 서버 프로세스 관리 유틸리티 선언 완료")


---
# 1. FastMCP 서버 개발 기초 & Pydantic 데이터 검증

FastMCP는 파이썬 데코레이터(`@app.tool`, `@app.resource`)를 통해 간결하게 MCP 표준 서버를 구축할 수 있는 현대적 프레임워크입니다.

### 1.1 In-Process FastMCP 서버 & 클라이언트 테스트
별도의 네트워크 포트 리스닝 없이, Python 프로세스 내에서 `Client(app)`로 즉시 연결하여 검증할 수 있습니다.


In [ ]:
import asyncio
from fastmcp import FastMCP
from fastmcp.client import Client

# 1. 데모 서버 인스턴스 생성
demo_server = FastMCP(
    name="in-process-demo",
    instructions="간단한 계산 및 사내 설정 조회를 제공하는 기본 데모 서버"
)

# 2. Tool Primitive 등록
@demo_server.tool(description="두 정수의 합을 계산합니다.")
def add_numbers(a: int, b: int) -> int:
    return a + b

@demo_server.tool(description="사내 지점별 날씨 및 환경 정보를 조회합니다.")
def get_branch_weather(branch: str) -> str:
    weather_map = {"본사": "맑음, 24°C", "판교연구소": "흐림, 22°C", "부산지사": "비, 20°C"}
    return weather_map.get(branch, f"{branch} 지점의 날씨 정보를 찾을 수 없습니다.")

# 3. Resource Primitive 등록 (읽기 전용 데이터)
@demo_server.resource("config://enterprise/settings")
def get_system_config() -> str:
    import json
    return json.dumps({"environment": "production", "region": "ap-northeast-2", "version": "2.4.0"}, indent=2)

# 4. In-Process 비동기 클라이언트 테스트
async def run_in_process_test():
    async with Client(demo_server) as client:
        print("--- [1] 노출 도구 목록 조회 ---")
        tools = await client.list_tools()
        for t in tools:
            print(f" • 도구명: {t.name:<20} | 설명: {t.description}")

        print("\n--- [2] add_numbers 도구 호출 ---")
        res1 = await client.call_tool("add_numbers", {"a": 45, "b": 55})
        print(f" • 결과: {res1.data}")

        print("\n--- [3] get_branch_weather 도구 호출 ---")
        res2 = await client.call_tool("get_branch_weather", {"branch": "판교연구소"})
        print(f" • 결과: {res2.data}")

# 비동기 실행 (Jupyter 환경 대응)
try:
    asyncio.run(run_in_process_test())
except RuntimeError:
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.get_event_loop().run_until_complete(run_in_process_test())


### 1.2 Pydantic 기반 입력 스키마 검증

실무 환경에서는 LLM이 전달하는 인자의 유효성을 Pydantic 모델(`BaseModel`, `Field`, `Enum`)로 엄격하게 검증하여 잘못된 쿼리가 백엔드 DB로 유입되는 것을 방지합니다.


In [ ]:
from enum import Enum
from pydantic import BaseModel, Field
from fastmcp import FastMCP

class BusinessRegion(str, Enum):
    SEOUL = "서울"
    PANGYO = "판교"
    BUSAN = "부산"
    GLOBAL = "해외"

class FinancialMetricQuery(BaseModel):
    """지점별 재무 지표 조회 요청 스키마"""
    region: BusinessRegion = Field(description="조회할 비즈니스 권역")
    year: int = Field(ge=2020, le=2030, description="조회 연도 (2020~2030)")
    quarter: int = Field(ge=1, le=4, description="분기 (1~4)")

schema_server = FastMCP(name="validated-financial-server")

@schema_server.tool(description="지정된 권역 및 분기의 집행 실적을 엄격한 스키마 검증 후 조회합니다.")
def query_regional_performance(query: FinancialMetricQuery) -> str:
    seed_val = hash(f"{query.region.value}_{query.year}_{query.quarter}")
    import random
    random.seed(seed_val)
    revenue = random.randint(50, 200) * 100000000
    return f"📊 [{query.region.value}] {query.year}년 Q{query.quarter} 매출 실적: {revenue:,}원"

In [ ]:
print("=== [정상 요청 테스트] ===")
valid_req = FinancialMetricQuery(region="판교", year=2026, quarter=2)
print(query_regional_performance(valid_req))


In [ ]:
print("\n=== [비정상 요청 스키마 차단 테스트] ===")
try:
    FinancialMetricQuery(region="뉴욕", year=2019, quarter=5)
except Exception as e:
    print(f"❌ Pydantic 유효성 검증 실패 (차단됨):\n{e}")


---
# 2. FastMCP 엔터프라이즈 인증/인가 (Auth) 아키텍처

엔터프라이즈 환경에서 데이터 소스 격리와 접근 권한 통제는 필수입니다.
FastMCP의 3단계 보안 패턴을 실습합니다:
1. **StaticTokenVerifier**: 토큰별 Scope(`read`, `write`, `admin`) 통제
2. **금융권 Custom Claims 인가**: 부서(`department`), 직급(`level`) 비즈니스 조건 검증
3. **가상 IdP RSA 비대칭키 JWT**: Keycloak/Azure AD와 동일한 JWT 발급 및 공개키 검증


### 2.1 StaticTokenVerifier (Scope 기반 권한 제어)
`secure_mcp_server.py`는 `admin-token-001`(모든 권한)과 `analyst-token-002`(read 전용)를 검증합니다.


In [ ]:
SECURE_PORT = find_available_port(6100, 6200)
SECURE_URL = start_mcp_server(
    name="Secure-MCP",
    script_path="example_mcp/secure_mcp_server.py",
    port=SECURE_PORT
)

In [ ]:
from fastmcp.client.auth import BearerAuth

async def test_static_auth():
    print("=== [1] Admin 토큰 테스트 (read + write + delete) ===")
    async with Client(SECURE_URL, auth=BearerAuth("admin-token-001")) as client:
        r1 = await client.call_tool("query_data", {"table": "sales"})
        print(f" • query_data:   ✅ {r1.data[:45]}...")
        r2 = await client.call_tool("update_record", {"record_id": 101, "field": "status", "new_value": "approved"})
        print(f" • update_record:✅ {r2.data}")

    print("\n=== [2] Analyst 토큰 테스트 (read만 허용) ===")
    async with Client(SECURE_URL, auth=BearerAuth("analyst-token-002")) as client:
        r1 = await client.call_tool("query_data", {"table": "sales"})
        print(f" • query_data:   ✅ {r1.data[:45]}...")
        try:
            await client.call_tool("update_record", {"record_id": 101, "field": "status", "new_value": "hacked"})
        except Exception as e:
            print(f" • update_record:❌ {type(e).__name__} (예상대로 권한 거부됨 403)")

    print("\n=== [3] 미인증 클라이언트 요청 ===")
    try:
        async with Client(SECURE_URL) as client:
            await client.call_tool("query_data", {"table": "sales"})
    except Exception as e:
        print(f" • 무인증 요청:  ❌ {type(e).__name__} (401 Unauthorized 차단)")

# Jupyter 비동기 실행
await test_static_auth()


### 2.2 금융권 Custom Claims 인가 (부서 및 직급 통제)

`custom_auth_server.py`는 `AuthContext.token.claims`로부터 `department` 및 `level`을 추출하여 비즈니스 권한을 통제하는 팩토리 함수를 사용합니다.


In [ ]:
CUSTOM_AUTH_PORT = find_available_port(6201, 6300)
CUSTOM_AUTH_URL = start_mcp_server(
    name="Custom-Auth-MCP",
    script_path="example_mcp/custom_auth_server.py",
    port=CUSTOM_AUTH_PORT
)

In [ ]:
async def test_custom_claims():
    print("=== [1] 재무팀 매니저 (department=finance, level=4) ===")
    async with Client(CUSTOM_AUTH_URL, auth=BearerAuth("finance-mgr-token")) as client:
        r1 = await client.call_tool("get_financial_report", {"period": "2026-Q1"})
        print(f" • 재무보고서:   ✅ {r1.data}")
        r2 = await client.call_tool("get_executive_dashboard", {})
        print(f" • 임원대시보드: ✅ {r2.data}")
        try:
            await client.call_tool("get_sales_pipeline", {})
        except Exception as e:
            print(f" • 영업파이프라인:❌ {type(e).__name__} (finance 부서 접근 차단)")

    print("=== [2] 영업팀 분석가 (department=sales, level=2) ===")
    async with Client(CUSTOM_AUTH_URL, auth=BearerAuth("sales-analyst-token")) as client:
        r1 = await client.call_tool("get_sales_pipeline", {})
        print(f" • 영업파이프라인:✅ {r1.data}")
        try:
            await client.call_tool("get_executive_dashboard", {})
        except Exception as e:
            print(f" • 임원대시보드: ❌ {type(e).__name__} (level 2 < 3 직급 미달 차단)")

try:
    asyncio.run(test_custom_claims())
except RuntimeError:
    asyncio.get_event_loop().run_until_complete(test_custom_claims())


### 2.3 가상 IdP 기반 RSA 비대칭키 JWT 서명 및 검증

실무 환경(Keycloak / Azure AD / Okta)과 동일하게 비대칭 키(RSA) 쌍을 생성하여, IdP 관점에서 JWT를 서명 발급하고 MCP 서버는 공개키 파일로 서명을 검증합니다.


In [ ]:
from fastmcp.server.auth.providers.jwt import RSAKeyPair
import os

# ⚠️ 서버의 jwt_mcp_server.py와 정확히 일치해야 합니다.
ISSUER = "https://auth.example-corp.com"
AUDIENCE = "mcp-enterprise-server"

# 1. RSA 비대칭 키 쌍 생성
key_pair = RSAKeyPair.generate()

# 2. 공개키를 로컬 파일로 저장 (서버가 검증기로 사용)
PUBLIC_KEY_FILE = os.path.abspath("jwt_public_key.txt")
with open(PUBLIC_KEY_FILE, "w", encoding="utf-8") as f:
    f.write(key_pair.public_key)

# 3. IdP 역할: 비공개키로 토큰 서명 생성
admin_jwt = key_pair.create_token(
    subject="admin-user-001",
    issuer=ISSUER,
    audience=AUDIENCE,
    scopes=["read", "write", "admin"],
    additional_claims={"department": "IT", "role": "admin", "level": 5}
)

analyst_jwt = key_pair.create_token(
    subject="analyst-user-002",
    issuer=ISSUER,
    audience=AUDIENCE,
    scopes=["read"],
    additional_claims={"department": "finance", "role": "analyst", "level": 2}
)

print(f"✅ RSA 공개키 저장 완료: {PUBLIC_KEY_FILE}")
print(f" • Admin JWT:   {admin_jwt[:45]}...")
print(f" • Analyst JWT: {analyst_jwt[:45]}...")


In [ ]:
JWT_PORT = find_available_port(6301, 6400)
JWT_URL = start_mcp_server(
    name="JWT-MCP",
    script_path="example_mcp/jwt_mcp_server.py",
    port=JWT_PORT,
    extra_args=["--public-key-file", PUBLIC_KEY_FILE]
)


In [ ]:
async def test_jwt_auth():
    print("=== [1] Admin JWT 비대칭 서명 검증 (read + write) ===")
    async with Client(JWT_URL, auth=BearerAuth(admin_jwt)) as client:
        r1 = await client.call_tool("secure_query", {"table": "sales"})
        print(f" • secure_query:  ✅ {r1.data}")
        r2 = await client.call_tool("secure_update", {"record_id": 1, "value": "Production-Active"})
        print(f" • secure_update: ✅ {r2.data}")

    print("=== [2] Analyst JWT (write 스코프 없음) ===")
    async with Client(JWT_URL, auth=BearerAuth(analyst_jwt)) as client:
        r1 = await client.call_tool("secure_query", {"table": "hr"})
        print(f" • secure_query:  ✅ {r1.data}")
        try:
            await client.call_tool("secure_update", {"record_id": 1, "value": "x"})
        except Exception as e:
            print(f" • secure_update: ❌ {type(e).__name__} (서명은 유효하나 write 권한 없음)")

try:
    asyncio.run(test_jwt_auth())
except RuntimeError:
    asyncio.get_event_loop().run_until_complete(test_jwt_auth())


---
# 3. Server Composition (엔터프라이즈 게이트웨이)

마이크로서비스 패턴에 따라 부서별 독립 FastMCP 서버(`sales`, `hr`, `finance`)를 구축한 뒤, 최상위 `Enterprise MCP Gateway`에 `gateway.mount(server, namespace='...')`로 합성합니다.


In [ ]:
sales_server = FastMCP("Sales-Service")
@sales_server.tool(description="분기별 매출 보고서 요약")
def get_sales_report(quarter: str) -> str:
    return {"Q1": "Q1: 150억원 (+12%)", "Q2": "Q2: 180억원 (+8%)"}.get(quarter, "데이터 없음")

hr_server = FastMCP("HR-Service")
@hr_server.tool(description="부서별 인원 현황 조회")
def get_headcount(department: str = "전체") -> str:
    data = {"개발본부": 140, "영업본부": 45, "경영지원": 20, "전체": 205}
    return f"{department}: {data.get(department, 0)}명"

finance_server = FastMCP("Finance-Service")
@finance_server.tool(description="부서별 예산 집행 현황")
def get_budget_status(department: str) -> str:
    budgets = {"개발본부": (80, 52), "영업본부": (30, 22)}
    t, u = budgets.get(department, (0, 0))
    return f"{department}: {t}억원 배정 중 {u}억원 집행 완료" if t else "데이터 없음"

# Enterprise Gateway로 합성
gateway = FastMCP("Enterprise-MCP-Gateway")
gateway.mount(sales_server, namespace="sales")
gateway.mount(hr_server, namespace="hr")
gateway.mount(finance_server, namespace="finance")

print("✅ Enterprise MCP Gateway 합성 완료")
print(" • 네임스페이스 도구: sales_get_sales_report, hr_get_headcount, finance_get_budget_status")


In [ ]:
async def test_composed_gateway():
    async with Client(gateway) as client:
        tools = await client.list_tools()
        print(f"📋 게이트웨이가 노출하는 합성 도구 목록 ({len(tools)}개):")
        for t in tools:
            print(f" • {t.name:<30} | {t.description}")

        print("🚀 네임스페이스 라우팅 호출 검증:")
        res_sales = await client.call_tool("sales_get_sales_report", {"quarter": "Q1"})
        print(f" • sales:   {res_sales.content[0].text if hasattr(res_sales, 'content') else res_sales.data}")

        res_hr = await client.call_tool("hr_get_headcount", {"department": "개발본부"})
        print(f" • hr:      {res_hr.content[0].text if hasattr(res_hr, 'content') else res_hr.data}")

        res_fin = await client.call_tool("finance_get_budget_status", {"department": "개발본부"})
        print(f" • finance: {res_fin.content[0].text if hasattr(res_fin, 'content') else res_fin.data}")

try:
    asyncio.run(test_composed_gateway())
except RuntimeError:
    asyncio.get_event_loop().run_until_complete(test_composed_gateway())


---
# 4. 연동 방식 A: `langchain_mcp_adapters` 정적 도구 바인딩

`langchain_mcp_adapters`의 `MultiServerMCPClient`는 원격 MCP 서버들에 접속하여 제공하는 모든 도구를 일괄 수집한 뒤 LangChain `BaseTool`로 변환합니다.

### 4.1 Basic & Finance MCP 서버 기동


In [ ]:
!pip install langchain_mcp_adapters

In [ ]:
BASIC_PORT = find_available_port(6401, 6500)
BASIC_URL = start_mcp_server(
    name="Basic-Demo-MCP",
    script_path="example_mcp/basic_mcp_server.py",
    port=BASIC_PORT
)

FINANCE_PORT = find_available_port(6501, 6600)
FINANCE_URL = start_mcp_server(
    name="Finance-Tools-MCP",
    script_path="example_mcp/finance_mcp_server.py",
    port=FINANCE_PORT
)


### 4.2 MultiServerMCPClient로 도구 일괄 바인딩 & `create_agent`


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

# 1. MCP 서버 엔드포인트 등록
mcp_client = MultiServerMCPClient({
    "basic-server": {
        "transport": "streamable_http",
        "url": BASIC_URL
    },
    "finance-server": {
        "transport": "streamable_http",
        "url": FINANCE_URL
    }
})

# 2. 모든 서버로부터 도구 일괄 로드
static_mcp_tools = await mcp_client.get_tools()

print(f"📦 정적 바인딩된 도구 목록 ({len(static_mcp_tools)}개):")
for t in static_mcp_tools:
    print(f" • {t.name:<20} | {t.description}")

# 3. LangChain 최신 권장 표준 create_agent 구성
static_agent = create_agent(
    model="openai:gpt-4o",
    system_prompt="""당신은 금융 데이터 분석 에이전트입니다.
사용자가 특정 종목의 주가나 정보를 요청하면 등록된 도구(stock_data 등)를 적극 활용하여 사실에 기반해 명확히 답변하세요.""",
    tools=static_mcp_tools,
    checkpointer=MemorySaver()
)

print("\n✅ static_agent 생성 완료")


In [ ]:
# 정적 주입 에이전트 실행 테스트
query = "애플(AAPL)과 마이크로소프트(MSFT)의 최근 주가 변동 현황을 요약해줘."

response = await static_agent.ainvoke(
    {"messages": [HumanMessage(content=query)]},
    config={"configurable": {"thread_id": "thread_static_1"}}
)

for msg in response["messages"]:
    msg.pretty_print()


### 💡 정적 도구 주입 방식의 한계 (Tool Explosion Problem)

| 문제점 | 세부 원인 및 실무적 영향 |
|:---|:---|
| **Context Bloat (토큰 낭비)** | 사내에 10개 MCP 서버가 존재하고 서버당 10개 도구가 있다면 **100개 도구의 스키마 전체가 매 턴마다 시스템 프롬프트에 주입**됩니다. (초기 컨텍스트만 수만 토큰 소모) |
| **Tool Confusion (도구 환각)** | 도구 수가 많아질수록 LLM이 유사한 이름의 엉뚱한 도구를 호출하거나 잘못된 파라미터를 조합할 확률이 급격히 증가합니다. |
| **서버 확장 시 코드 재배포** | 새로운 부서 서버가 신설될 때마다 에이전트 코드의 `MultiServerMCPClient` 설정을 수정하고 에이전트를 재배포해야 합니다. |


---
# 5. 연동 방식 B: Anthropic Skills 패턴 + 범용 원시 도구 (차세대 아키텍처)

도구를 프롬프트에 직접 바인딩하지 않고, **"도구를 다루는 매뉴얼(Skill)"**과 **"원시 실행기(Primitive Tools)"**를 분리합니다.

```mermaid
graph LR
    subgraph FrontierAgent ["Frontier Agent (create_agent)"]
        Tools["범용 원시 도구 (tools/common.py)<br/>• file_read<br/>• bash_command<br/>• glob_search"]
    end

    subgraph SkillsDir ["로컬 skills/ 디렉토리"]
        Skill1["skills/jupyter-notebook/<br/>(SKILL.md, new_notebook.py)"]
        Skill2["skills/mcp/<br/>(SKILL.md, list_tools.py, execute_tool.py)"]
    end

    subgraph RemoteServers ["독립 FastMCP 서버들"]
        S1["Finance Server (:6501)"]
        S2["Basic Server (:6401)"]
        S3["N개의 신규 서버..."]
    end

    Tools -->|"1. glob_search"| SkillsDir
    Tools -->|"2. file_read(SKILL.md)"| Skill2
    Tools -->|"3. bash_command(CLI 실행)"| RemoteServers
```

### 5.1 로컬 `skills/` 생태계 및 범용 도구 확인

에이전트는 비즈니스 도구를 모르며, 오직 [`tools/common.py`](file:///h:/%EB%82%B4%20%EB%93%9C%EB%9D%BC%EC%9D%B4%EB%B8%8C/work_memory/contexts/Lectures/handson/09_%EC%97%90%EC%9D%B4%EC%A0%84%ED%8B%B1_RAG_%EC%8B%A4%EB%AC%B4/2.MCP/tools/common.py)에 정의된 **3가지 범용 도구(`glob_search`, `file_read`, `bash_command`)**만 장착합니다.


In [ ]:
from tools.common import glob_search, file_read, bash_command

primitive_tools = [glob_search, file_read, bash_command]

print(f"🛠️ 에이전트 장착 범용 도구 ({len(primitive_tools)}개):")
for t in primitive_tools:
    print(f" • {t.name:<15} : {t.description[:60]}...")


### 5.2 Skills 기반 Frontier Agent 구축

시스템 프롬프트는 에이전트에게 비즈니스 스키마를 주입하지 않고, **"작업 해결을 위한 점진적 탐색 3단계 지침"**만을 제공합니다.


In [ ]:
SKILL_AGENT_PROMPT = """당신은 Anthropic Agent Skills 표준을 준수하는 지능형 에이전트입니다.
당신은 사전에 고정된 비즈니스 도구를 가지고 있지 않으며, 파일시스템의 `skills/` 디렉토리에 위치한 스킬들을 동적으로 탐색하고 실행하여 문제를 해결해야 합니다.

[작업 수행 프로토콜 - 점진적 공개(Progressive Disclosure)]
1. [스킬 탐색]: 먼저 `glob_search`를 실행하여 `skills/**/SKILL.md` 경로들을 찾고 현재 사용 가능한 스킬 목록을 파악하세요.
2. [스킬 매뉴얼 습득]: 질문을 해결하는 데 적합한 스킬을 찾았으면, `file_read`로 해당 스킬의 `SKILL.md` 전문을 읽어 CLI 스크립트의 인자 규격과 실행 예시를 확인하세요.
3. [도구 탐색 및 실행]:
   - MCP 서버의 도구가 필요하다면 `bash_command`로 `python skills/mcp/scripts/list_tools.py --url <URL>`을 실행하여 도구 목록을 조회하세요.
   - 적절한 도구를 찾았으면 `bash_command`로 `python skills/mcp/scripts/execute_tool.py --url <URL> --tool <TOOL_NAME> --args '<JSON_STRING>'`을 실행하여 데이터를 가져오세요.
4. [최종 답변]: 수집된 데이터를 종합하여 사용자에게 명확하고 통찰력 있는 답변을 제공하세요.
"""

skill_agent = create_agent(
    model="openai:gpt-4o",
    system_prompt=SKILL_AGENT_PROMPT,
    tools=primitive_tools,
    checkpointer=MemorySaver()
)

print("✅ Skills 기반 Frontier 에이전트 생성 완료 (프롬프트 토큰 최소화 달성)")


### 5.3 실전 테스트: 자율적 스킬 탐색 & 엔비디아/애플 주가 분석

에이전트가 `glob_search` -> `file_read("skills/mcp/SKILL.md")` -> `bash_command(list_tools)` -> `bash_command(execute_tool)` 라이프사이클을 스스로 거치는지 확인합니다.


In [ ]:
query = f"""현재 로컬에 구동 중인 금융 서버({FINANCE_URL})를 확인하여,
엔비디아(NVDA)와 테슬라(TSLA)의 최근 주가 지표를 조회하고 간략한 투자 비교 보고서를 작성해줘."""

response = await skill_agent.ainvoke(
    {"messages": [HumanMessage(content=query)]},
    config={"configurable": {"thread_id": "thread_skills_1"}}
)

print("" + "="*80)
print("🎯 Skills 기반 에이전트 실행 궤적 (Execution Trace):")
print("="*80)

for msg in response["messages"]:
    msg.pretty_print()


---
# 6. 세션 정리 및 실습 프로세스 종료

실습이 완료된 후 백그라운드에서 실행 중인 모든 FastMCP 서버 프로세스를 안전하게 종료합니다.


In [ ]:
# 기동된 모든 백그라운드 FastMCP 서버 정상 종료
stop_all_servers()
print("🎉 실습 완료: 모든 서버 프로세스가 성공적으로 정리되었습니다.")
